<a href="https://colab.research.google.com/github/potato-yen/114-2-Programming-Language/blob/main/HW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1jR3qRQr2ZvWYKNuv8wen_-eTZWdc5a-LLvH7iymn2zw/edit?usp=sharing

In [53]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [54]:
import pandas as pd
# read data and put it in a dataframe
# 在 google 工作表載入 gsheets
gsheets = gc.open_by_url('https://docs.google.com/spreadsheets/d/1Etj3aomNgs1hroV3JwgRMNMVQ2_BIyKwChgtPRnYxic/edit?usp=sharing')

In [55]:
# 從 gsheets 的 All-whiteboard-device 載入 sheets
sheets = gsheets.worksheet('工作表1').get_all_values()
# 將 sheets1 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(sheets[1:], columns=sheets[0])
# 取得最前面的5筆資料
df.head()

,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註
0,2022-01-02,10:13,食物,food,200,Amethus,7-11,Apple pay,炒麵


In [56]:
import datetime

In [58]:
import datetime

# 定義購物分類
categories = {
    1: '食物',
    2: '飲料',
    3: '交通',
    4: '娛樂',
    5: '居住'
}

# 讓使用者輸入資料
while True:
    date_str = input("請輸入日期 (格式：YYYY-MM-DD): ")
    try:
        datetime.datetime.strptime(date_str, '%Y-%m-%d')
        break
    except ValueError:
        print("格式錯誤，請重新輸入，例如：2023-01-01")

while True:
    time_str = input("請輸入時間 (格式：HH:MM): ")
    try:
        datetime.datetime.strptime(time_str, '%H:%M')
        break
    except ValueError:
        print("格式錯誤，請重新輸入，例如：14:30")

item = input("請輸入品項: ")
amount = float(input("請輸入金額: "))

# 讓使用者選擇分類
while True:
    print("請選擇分類 (輸入數字):")
    for key, value in categories.items():
        print(f"  {key}: {value}")
    print("  6: 其他")
    category_choice = input("您的選擇: ")
    try:
        choice_int = int(category_choice)
        if choice_int in categories:
            category = categories[choice_int]
            break
        elif choice_int == 6:
            category = input("請輸入其他分類名稱: ")
            break
        else:
            print("無效的選擇，請重新輸入。")
    except ValueError:
        print("請輸入數字。")

# 讓使用者輸入備註，限制20個字元，可為空值
remark = input("請輸入備註 (可選，最多20字元): ")
while len(remark) > 20:
    print("備註長度超過20個字元，請重新輸入。")
    remark = input("請輸入備註 (可選，最多20字元): ")

請輸入日期 (格式：YYYY-MM-DD): 2022-02-01
請輸入時間 (格式：HH:MM): 20:21
請輸入品項: 蘿蔔糕
請輸入金額: 50
請選擇分類 (輸入數字):
  1: 食物
  2: 飲料
  3: 交通
  4: 娛樂
  5: 居住
  6: 其他
您的選擇: 1
請輸入備註 (可選，最多20字元): 真好吃


In [59]:
date_str

'2022-02-01'

In [60]:
time_str

'20:21'

In [61]:
item

'蘿蔔糕'

In [62]:
amount

50.0

In [63]:
# 格式化日期和時間為標準格式
formatted_date = datetime.datetime.strptime(date_str, '%Y-%m-%d').strftime('%Y-%m-%d')
formatted_time = datetime.datetime.strptime(time_str, '%H:%M').strftime('%H:%M')

# 創建一個新的 DataFrame 來代表你要新增的資料，包含所有原始欄位並填補預設值
new_data = pd.DataFrame([{
    '日期': formatted_date,
    '時間': formatted_time,
    '分類': category,
    '品項': item,
    '金額': amount,
    '付款人': 'Amethus',
    '地點': '7-11',
    '支付方式': 'Apple pay',
    '備註': remark
}])

# 使用 concat() 將新資料合併到舊的 df 中
df = pd.concat([df, new_data], ignore_index=True)

In [64]:
df

,日期,時間,分類,品項,金額,付款人,地點,支付方式,備註
0,2022-01-02,10:13,食物,food,200,Amethus,7-11,Apple pay,炒麵
1,2022-02-01,20:21,食物,蘿蔔糕,50.0,Amethus,7-11,Apple pay,真好吃


In [65]:
# 將 DataFrame 轉換成列表的列表 (list of lists)，這是 gspread 支援的資料格式
data_to_write = df.values.tolist()

# 第一步：取得工作表物件
worksheet = gsheets.worksheet('工作表1')

# 第二步：將資料寫入工作表
worksheet.append_rows(values=data_to_write, value_input_option='USER_ENTERED')
print("資料已成功寫入 Google 工作表！")

資料已成功寫入 Google 工作表！


In [66]:
# 在將 df 轉換為列表之前，先將所有 NaN 值替換為空字串
df_cleaned = df.fillna('')

# 將 DataFrame 轉換成列表的列表 (list of lists)，這是 gspread 支援的資料格式
data_to_write = df_cleaned.values.tolist()

# 第一步：取得工作表物件
worksheet = gc.open_by_url('https://docs.google.com/spreadsheets/d/1Etj3aomNgs1hroV3JwgRMNMVQ2_BIyKwChgtPRnYxic/edit?usp=sharing').worksheet('工作表1')

# 清除現有內容，準備寫入更新後的整個 DataFrame
# 注意: 這會清除工作表1的所有內容，請確認這是您想要的操作
worksheet.clear()

# 將 df_cleaned 的標頭寫入工作表
worksheet.append_row(df_cleaned.columns.tolist())

# 第二步：將資料寫入工作表 (不包含標頭)
worksheet.append_rows(values=data_to_write, value_input_option='USER_ENTERED')
print("資料已成功寫入 Google 工作表！")

資料已成功寫入 Google 工作表！
